8. Testing new features
   Some features such as number of the projects in the same location and to_package are likely to play an important role in delays. Therefore, it worth it to asses their role and compare the results with the old data.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA, MODELS, METRICS = ROOT/'data/processed/final_dataset.csv', ROOT/'models', ROOT/'results/metrics'
MODELS.mkdir(parents=True, exist_ok=True); METRICS.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
y = df.pop('overdue_label')

# 1. Base Feature Selection 
drop = ['description', 'target', 'priority', 'cause', 'documents', 'comments', 'images', 'association']
X = df.drop(columns=[c for c in drop if c in df], errors='ignore')

# 2. Train-Test Split (Preventing Data Leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Experimental Datasets
X_train_exp = X_train.copy()
X_test_exp = X_test.copy()

# ---------------------------------------------------------
# Feature Engineering (Experimental Features)
# ---------------------------------------------------------

# Feature 1: Count of tasks in same location (Overall)
if 'location' in X_train.columns:
    loc_counts = X_train_exp['location'].value_counts()
    X_train_exp['tasks_in_same_location'] = X_train_exp['location'].map(loc_counts).fillna(1)
    X_test_exp['tasks_in_same_location'] = X_test_exp['location'].map(loc_counts).fillna(1)

# Feature 2 & 3: Monthly Location Tasks & Workload Ratio
if 'location' in X_train.columns and 'created' in X_train.columns:
   # Converting to datetime
    X_train_exp['created_dt'] = pd.to_datetime(X_train_exp['created'], errors='coerce')
    X_test_exp['created_dt'] = pd.to_datetime(X_test_exp['created'], errors='coerce')

    X_train_exp['year_month'] = X_train_exp['created_dt'].dt.to_period('M').astype(str)
    X_test_exp['year_month'] = X_test_exp['created_dt'].dt.to_period('M').astype(str)

    # 1. The number of tasks for each location in the current month
    loc_month_counts = X_train_exp.groupby(['location', 'year_month']).size()

    train_loc_month_keys = list(zip(X_train_exp['location'], X_train_exp['year_month']))
    test_loc_month_keys = list(zip(X_test_exp['location'], X_test_exp['year_month']))

    X_train_exp['tasks_in_same_loc_month'] = [loc_month_counts.get(k, 1) for k in train_loc_month_keys]
    X_test_exp['tasks_in_same_loc_month'] = [loc_month_counts.get(k, 1) for k in test_loc_month_keys]

    # 2. Calculating the number of tasks in each month (Only on Train Data)
    loc_monthly_avg = (
        X_train_exp.groupby(['location', 'year_month'])
        .size()
        .groupby('location')
        .mean()
    )

    # Maping location avarage number monthly
    train_loc_avg = X_train_exp['location'].map(loc_monthly_avg).fillna(1)
    test_loc_avg = X_test_exp['location'].map(loc_monthly_avg).fillna(1)

    # 3. Calculating (Workload Ratio)
    X_train_exp['loc_workload_ratio'] = X_train_exp['tasks_in_same_loc_month'] / train_loc_avg
    X_test_exp['loc_workload_ratio'] = X_test_exp['tasks_in_same_loc_month'] / test_loc_avg

    # Changing Inf with NaN 
    X_train_exp['loc_workload_ratio'] = X_train_exp['loc_workload_ratio'].replace([np.inf, -np.inf], 1.0).fillna(1.0)
    X_test_exp['loc_workload_ratio'] = X_test_exp['loc_workload_ratio'].replace([np.inf, -np.inf], 1.0).fillna(1.0)

    # temporary removal
    X_train_exp = X_train_exp.drop(columns=['created_dt', 'year_month'], errors='ignore')
    X_test_exp = X_test_exp.drop(columns=['created_dt', 'year_month'], errors='ignore')

# Feature 4: Out-of-Fold / Train-only Target Encoding for 'to_package'
if 'to_package' in X_train.columns:
    global_mean = y_train.mean()
    package_rates = y_train.groupby(X_train['to_package']).mean()
    
    X_train_exp['package_overdue_rate'] = X_train_exp['to_package'].map(package_rates).fillna(global_mean)
    X_test_exp['package_overdue_rate'] = X_test_exp['to_package'].map(package_rates).fillna(global_mean)

# removing from Baseline و Experimental
if 'created' in X_train.columns:
    X_train = X_train.drop(columns=['created'], errors='ignore')
    X_test = X_test.drop(columns=['created'], errors='ignore')
    X_train_exp = X_train_exp.drop(columns=['created'], errors='ignore')
    X_test_exp = X_test_exp.drop(columns=['created'], errors='ignore')

if 'to_package' in X_train.columns:
    X_train = X_train.drop(columns=['to_package'], errors='ignore')
    X_test = X_test.drop(columns=['to_package'], errors='ignore')

# 3. Preprocessors
cat = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num = [c for c in X_train.columns if c not in cat]

cat_exp = X_train_exp.select_dtypes(include=['object', 'category']).columns.tolist()
num_exp = [c for c in X_train_exp.columns if c not in cat_exp]

pre = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num), 
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat)
])

pre_exp = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_exp), 
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_exp)
])

# 4. Models Setup
ratio = (len(y_train) - sum(y_train)) / sum(y_train)

models = {
    'logistic_regression': LogisticRegression(max_iter=2000, class_weight='balanced'),
    'random_forest': RandomForestClassifier(n_estimators=400, min_samples_leaf=3, class_weight='balanced', random_state=42, n_jobs=-1),
    'xgboost': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, scale_pos_weight=ratio, random_state=42, n_jobs=-1),
    'lightgbm': LGBMClassifier(n_estimators=300, max_depth=6, learning_rate=0.05, class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1),
    'catboost': CatBoostClassifier(iterations=300, depth=6, learning_rate=0.05, auto_class_weights='Balanced', random_state=42, verbose=0)
}

cv = StratifiedKFold(5, shuffle=True, random_state=42)

# 5. Cross-Validation
rows, rows_exp = [], []
for name, model in models.items():
    # Baseline
    pipe = Pipeline([('preprocessor', pre), ('model', model)])
    score = cross_validate(pipe, X_train, y_train, cv=cv, scoring=['roc_auc','precision','recall','f1','accuracy'], n_jobs=-1)
    rows.append({'model': name, **{k.replace('test_','cv_'): v.mean() for k, v in score.items() if k.startswith('test_')}})
    
    # Experimental
    pipe_exp = Pipeline([('preprocessor', pre_exp), ('model', model)])
    score_exp = cross_validate(pipe_exp, X_train_exp, y_train, cv=cv, scoring=['roc_auc','precision','recall','f1','accuracy'], n_jobs=-1)
    rows_exp.append({'model': name, **{k.replace('test_','exp_cv_'): v.mean() for k, v in score_exp.items() if k.startswith('test_')}})

metrics = pd.DataFrame(rows).sort_values('cv_f1', ascending=False)
metrics_exp = pd.DataFrame(rows_exp).sort_values('exp_cv_f1', ascending=False)

# Compare Metrics
comparison = metrics.merge(metrics_exp, on='model')
print(comparison[['model', 'cv_f1', 'exp_cv_f1', 'cv_precision', 'exp_cv_precision', 'cv_recall', 'exp_cv_recall']])

# Save best improved model and dataset holdout artifacts
best_name_exp = metrics_exp.iloc[0]['model']
best_exp = Pipeline([('preprocessor', pre_exp), ('model', models[best_name_exp])])
best_exp.fit(X_train_exp, y_train)

joblib.dump(best_exp, MODELS/'overdue_risk_new_features_added_model.joblib')
joblib.dump({
    'X_test': X_test_exp, 
    'y_test': y_test, 
    'feature_columns': X_train_exp.columns.tolist()
}, MODELS/'evaluation_new_features_added_holdout.joblib')

# Save metric CSVs
metrics_exp.to_csv(METRICS/'cross_validation_new_features_added_metrics.csv', index=False)

print(f"\nSuccessfully saved updated best model ({best_name_exp}) with engineered features!")

                 model     cv_f1  exp_cv_f1  cv_precision  exp_cv_precision  \
0             lightgbm  0.669394   0.684702      0.521291          0.545880   
1              xgboost  0.634350   0.674145      0.479280          0.531088   
2             catboost  0.623670   0.626296      0.466621          0.468649   
3  logistic_regression  0.600300   0.615941      0.441745          0.460307   
4        random_forest  0.501632   0.530418      0.335900          0.364658   

   cv_recall  exp_cv_recall  
0   0.935484       0.920503  
1   0.940029       0.926551  
2   0.941533       0.944507  
3   0.937055       0.931040  
4   0.991000       0.973000  

Successfully saved updated best model (lightgbm) with engineered features!
